# 01b · Adquisición complementaria — Grupo 17
**Paso 1b del Sprint 2** · Snapshot aparte: `g17-phishing-complemento-v1`. El snapshot v1 no se toca.

| Fuente | Rol | Licencia |
|---|---|---|
| Nazario, archivos anuales 2015–2025 | 3.er origen de la clase phishing (reciente) | CC BY 4.0 |
| phishing_pot (rf-peixoto), 2022–2026 | 4.º origen de la clase phishing (muy reciente) | CC BY-NC 4.0 (uso académico) |
| PhishTank snapshot (GitHub, 2008–2018) | Validación externa de la rama URL (solo positivos) | Términos de PhishTank |

Se excluyen a propósito: `private-phishing4.mbox` (el autor lo marca como privado) y los mbox 2005–2007,
que probablemente se solapan con `Nazario.csv` de Kaggle.

**Criterios de cierre:** (1) manifest y verificación sin discrepancias, (2) conteo de correos por archivo,
(3) commit de GitHub fijado, (4) pruebas de `ioc_utils` en verde en este entorno,
(5) 0 textos con IOC vivo tras `defang_texto` sobre los 82.486 correos de v1.

In [1]:
# Celda 1: Entorno (misma lógica que 01)
import os, sys, json, hashlib, random, platform, shutil, re, mailbox, zipfile, subprocess
import datetime as dt
from pathlib import Path
import numpy as np, pandas as pd

SEED = 17
random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"

ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/PI_G17")
elif ENV == "kaggle":
    BASE = Path("/kaggle/working/PI_G17")
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))

SNAP_V1 = BASE / "data" / "raw" / "g17-phishing-raw-v1"
SNAPSHOT = "g17-phishing-complemento-v1"
RAW  = BASE / "data" / "raw" / SNAPSHOT
META = BASE / "data" / "meta"
for p in (RAW, META): p.mkdir(parents=True, exist_ok=True)
UA = {"User-Agent": "UEES-MIAR0545-Grupo17-research/1.0"}
print(f"Entorno: {ENV}\nRAW = {RAW}")

def sha256_archivo(path, bloque=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(bloque), b""): h.update(c)
    return h.hexdigest()

def inventario_hashes(raiz):
    return {p.relative_to(raiz).as_posix(): {"sha256": sha256_archivo(p), "bytes": p.stat().st_size}
            for p in sorted(raiz.rglob("*")) if p.is_file()}

def ahora_utc():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()

fuentes = {}

Mounted at /content/drive
Entorno: colab
RAW = /content/drive/MyDrive/PI_G17/data/raw/g17-phishing-complemento-v1


In [2]:
# Celda 2: Pruebas de ioc_utils en ESTE entorno (criterio 4)
# Acepta la carpeta de pruebas como test/ o tests/ (el repo usa tests/).
%pip install -q pytest requests
carpetas = [d for d in (BASE / "tests", BASE / "test") if d.exists()]
assert (BASE / "src" / "ioc_utils.py").exists(), "Falta src/ioc_utils.py en PI_G17"
assert carpetas, "Falta la carpeta test/ o tests/ en PI_G17"
r = subprocess.run([sys.executable, "-m", "pytest", "-q", str(carpetas[0])], capture_output=True, text=True, cwd=BASE)
print(r.stdout[-600:], r.stderr[-400:])
PRUEBAS_IOC = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else "sin salida"
sys.path.insert(0, str(BASE / "src"))
import ioc_utils

........................................................................ [ 22%]
........................................................................ [ 45%]
........................................................................ [ 68%]
........................................................................ [ 91%]
...........................                                              [100%]
315 passed in 2.95s
 


In [3]:
# Celda 3: Nazario 2015–2025
import requests
NAZ_URL = "https://monkey.org/~jose/phishing/"
DEST_NAZ = RAW / "nazario_anual"; DEST_NAZ.mkdir(parents=True, exist_ok=True)
html = requests.get(NAZ_URL, headers=UA, timeout=60).text
anuales = sorted(set(re.findall(r'href="(phishing-(\d{4}))"', html)))
anuales = [n for n, y in anuales if int(y) >= 2015]
for n in anuales + ["LICENSE.txt", "README.txt"]:
    dst = DEST_NAZ / n
    if not dst.exists():
        rr = requests.get(NAZ_URL + n, headers=UA, timeout=300); rr.raise_for_status()
        dst.write_bytes(rr.content)
fuentes["nazario_anual"] = {"origen": NAZ_URL, "archivos": anuales, "licencia": "CC BY 4.0 (README.txt del autor)",
                            "cita": "Nazario, J. Phishing Corpus", "descargado_utc": ahora_utc()}
print(anuales)

['phishing-2015', 'phishing-2016', 'phishing-2017', 'phishing-2018', 'phishing-2019', 'phishing-2020', 'phishing-2021', 'phishing-2022', 'phishing-2023', 'phishing-2024', 'phishing-2025']


In [4]:
# Celda 4: Repos de GitHub fijados por commit
import requests  # independiente de la Celda 3: si monkey.org falla, esta celda sigue
def commit_actual(repo, rama):
    try:
        rr = requests.get(f"https://api.github.com/repos/{repo}/commits/{rama}", headers=UA, timeout=30)
        rr.raise_for_status(); return rr.json()["sha"]
    except Exception as e:
        print(f"Aviso: no se pudo fijar commit de {repo} ({type(e).__name__}); se usa la rama"); return None

# phishing_pot: se guarda el ZIP sin descomprimir (la extracción es preparación, Paso 4)
POT_REPO = "rf-peixoto/phishing_pot"
sha_pot = commit_actual(POT_REPO, "main")
ref = sha_pot or "refs/heads/main"
dst = RAW / "phishing_pot" / f"phishing_pot-{(sha_pot or 'main')[:12]}.zip"; dst.parent.mkdir(parents=True, exist_ok=True)
if not dst.exists():
    rr = requests.get(f"https://codeload.github.com/{POT_REPO}/zip/{ref}", headers=UA, timeout=900); rr.raise_for_status()
    dst.write_bytes(rr.content)
fuentes["phishing_pot"] = {"origen": f"https://github.com/{POT_REPO}", "commit": sha_pot,
                           "licencia": "CC BY-NC 4.0", "descargado_utc": ahora_utc()}

# PhishTank snapshot (del documento de enlaces del equipo)
PT_REPO = "arvindeybram/phishing"
sha_pt = commit_actual(PT_REPO, "master")
dst = RAW / "phishtank_github" / "phishtank.csv"; dst.parent.mkdir(parents=True, exist_ok=True)
if not dst.exists():
    rr = requests.get(f"https://raw.githubusercontent.com/{PT_REPO}/{sha_pt or 'master'}/phishtank.csv", headers=UA, timeout=300)
    rr.raise_for_status(); dst.write_bytes(rr.content)
fuentes["phishtank_github"] = {"origen": f"https://github.com/{PT_REPO}", "commit": sha_pt,
                               "licencia": "Términos de PhishTank (Cisco Talos); citar la fuente", "descargado_utc": ahora_utc()}
print("pot:", sha_pot, "| phishtank:", sha_pt)

pot: 49f63777126b0bdb9eb1f6e770a5c3f9df2b0306 | phishtank: 619b0e2066838dc5780032028292ccfd5255b075


In [9]:
# Celda 5: Congelar o verificar
MANIFEST = META / f"manifest_{SNAPSHOT}.json"
if MANIFEST.exists():
    manifest = json.loads(MANIFEST.read_text())
    actual = inventario_hashes(RAW); esp = manifest["archivos"]
    verificacion = {"faltan": sorted(set(esp) - set(actual)),
                    "sobran": sorted(set(actual) - set(esp) - {"manifest.json"}),
                    "hash_distinto": sorted(k for k in set(esp) & set(actual) if esp[k]["sha256"] != actual[k]["sha256"])}
    verificacion["ok"] = not any(verificacion.values())
    print("MODO VERIFICACIÓN ->", verificacion)
else:
    manifest = {"snapshot": SNAPSHOT, "creado_utc": ahora_utc(), "entorno": ENV,
                "fuentes": fuentes, "archivos": inventario_hashes(RAW)}
    MANIFEST.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
    shutil.copy2(MANIFEST, RAW / "manifest.json")
    verificacion = None
    print(f"Manifest creado ({len(manifest['archivos'])} archivos). Vuelve a ejecutar esta celda para verificar.")

MODO VERIFICACIÓN -> {'faltan': [], 'sobran': [], 'hash_distinto': [], 'ok': True}


In [10]:
# Celda 6: Conteos (sin parsear contenido)
conteos = {}
for f in sorted((RAW / "nazario_anual").glob("phishing-*")):
    conteos[f.name] = len(mailbox.mbox(str(f)))
for z in sorted((RAW / "phishing_pot").glob("*.zip")):
    conteos[z.name] = sum(n.lower().endswith(".eml") for n in zipfile.ZipFile(z).namelist())
pt = pd.read_csv(RAW / "phishtank_github" / "phishtank.csv")
conteos["phishtank.csv"] = int(len(pt))
rango_pt = [str(pt["submission_time"].min())[:10], str(pt["submission_time"].max())[:10]]
conteos

{'phishing-2015': 307,
 'phishing-2016': 494,
 'phishing-2017': 325,
 'phishing-2018': 288,
 'phishing-2019': 243,
 'phishing-2020': 158,
 'phishing-2021': 101,
 'phishing-2022': 247,
 'phishing-2023': 419,
 'phishing-2024': 403,
 'phishing-2025': 481,
 'phishing_pot-49f63777126b.zip': 8614,
 'phishtank.csv': 32239}

In [11]:
# Celda 7: Validación de ioc_utils sobre el corpus real v1 (criterio 5)
# Mide también el % de correos con URL por subcorpus: primer dato del riesgo 2.
SUB = ["CEAS_08", "Enron", "Ling", "SpamAssasin", "Nazario", "Nigerian_Fraud"]
def leer(p):
    for enc in ("utf-8", "latin-1"):
        try: return pd.read_csv(p, encoding=enc, low_memory=False)
        except UnicodeDecodeError: pass
defang_ok, cobertura = {}, {}
for s in SUB:
    p = next(SNAP_V1.rglob(f"{s}.csv"))
    df = leer(p)
    txt = (df["subject"].fillna("").astype(str) + "\n" + df["body"].fillna("").astype(str))
    vivos = txt.map(lambda t: bool(ioc_utils.iocs_vivos(ioc_utils.defang_texto(t))))
    con_url = txt.map(lambda t: len(ioc_utils.urls_para_features(t)) > 0)
    defang_ok[s] = {"textos": int(len(txt)), "con_ioc_vivo_tras_defang": int(vivos.sum())}
    cobertura[s] = round(100 * float(con_url.mean()), 1)
print("Defang:", defang_ok); print("% correos con URL:", cobertura)

Defang: {'CEAS_08': {'textos': 39154, 'con_ioc_vivo_tras_defang': 3}, 'Enron': {'textos': 29767, 'con_ioc_vivo_tras_defang': 0}, 'Ling': {'textos': 2859, 'con_ioc_vivo_tras_defang': 0}, 'SpamAssasin': {'textos': 5809, 'con_ioc_vivo_tras_defang': 3}, 'Nazario': {'textos': 1565, 'con_ioc_vivo_tras_defang': 0}, 'Nigerian_Fraud': {'textos': 3332, 'con_ioc_vivo_tras_defang': 0}}
% correos con URL: {'CEAS_08': 67.5, 'Enron': 0.0, 'Ling': 0.0, 'SpamAssasin': 79.2, 'Nazario': 16.2, 'Nigerian_Fraud': 36.7}


In [12]:
# Celda 8: RESUMEN
print("===== RESUMEN (01b) =====")
print(json.dumps({
    "entorno": ENV, "pruebas_ioc_utils": PRUEBAS_IOC,
    "verificacion": verificacion, "n_archivos": len(manifest["archivos"]),
    "commits": {k: v.get("commit") for k, v in manifest["fuentes"].items() if "commit" in v},
    "nazario_archivos": manifest["fuentes"].get("nazario_anual", {}).get("archivos"),
    "conteos": conteos, "phishtank_rango": rango_pt,
    "defang_corpus_v1": defang_ok, "pct_con_url_v1": cobertura,
}, indent=1, ensure_ascii=False, default=str))

===== RESUMEN (01b) =====
{
 "entorno": "colab",
 "pruebas_ioc_utils": "315 passed in 2.95s",
 "verificacion": {
  "faltan": [],
  "sobran": [],
  "hash_distinto": [],
  "ok": true
 },
 "n_archivos": 15,
 "commits": {
  "phishing_pot": "49f63777126b0bdb9eb1f6e770a5c3f9df2b0306",
  "phishtank_github": "619b0e2066838dc5780032028292ccfd5255b075"
 },
 "nazario_archivos": [
  "phishing-2015",
  "phishing-2016",
  "phishing-2017",
  "phishing-2018",
  "phishing-2019",
  "phishing-2020",
  "phishing-2021",
  "phishing-2022",
  "phishing-2023",
  "phishing-2024",
  "phishing-2025"
 ],
 "conteos": {
  "phishing-2015": 307,
  "phishing-2016": 494,
  "phishing-2017": 325,
  "phishing-2018": 288,
  "phishing-2019": 243,
  "phishing-2020": 158,
  "phishing-2021": 101,
  "phishing-2022": 247,
  "phishing-2023": 419,
  "phishing-2024": 403,
  "phishing-2025": 481,
  "phishing_pot-49f63777126b.zip": 8614,
  "phishtank.csv": 32239
 },
 "phishtank_rango": [
  "2008-06-20",
  "2018-07-17"
 ],
 "defang_co